# SugarSense: Early Diabetes Risk Screening

## AI/ML Mini Project — Supervised Learning

### Objective

The objective of this project is to build a supervised machine learning model that predicts whether a patient is likely to have diabetes using clinical measurements.

### Dataset

Pima Indians Diabetes Database

### Problem Type

Binary Classification

### Target Variable

`Outcome`

- 0 = Not diabetic
- 1 = Diabetic

### Primary Evaluation Metrics

- ROC-AUC
- Recall

### Important Note

This project is an educational machine learning screening system and is not intended to provide a medical diagnosis.

## Project Overview

This project follows a complete supervised machine learning workflow:

1. Data Audit
2. Exploratory Data Analysis
3. Feature Engineering
4. Train/Test Split
5. Leakage-Safe Preprocessing Pipelines
6. Model Comparison
7. Hyperparameter Tuning
8. Final Test Evaluation
9. Threshold Tuning
10. Feature Importance
11. Final Conclusions

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
    cross_val_predict
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

RANDOM_STATE = 42

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

## Load the Dataset

The dataset is stored in CSV format. It contains patient clinical measurements and an `Outcome` column indicating whether diabetes was present.

In [ ]:
df = pd.read_csv("diabetes_screening_data.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

## Data Understanding

The dataset contains 768 patient records and 9 columns.

The features include pregnancies, glucose level, blood pressure, skin thickness, insulin, BMI, diabetes pedigree function, and age.

The `Outcome` column is the target variable. A value of 1 indicates that diabetes is present, while 0 indicates that diabetes is not present.

The objective of this project is to analyze these health-related features and build supervised machine learning models for early diabetes risk screening.


In [ ]:
print("Dataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

## Exploratory Data Analysis

Exploratory Data Analysis (EDA) is performed to understand the distribution of the variables and identify important patterns in the dataset.

We will examine the distribution of the target variable and visualize relationships between important health-related features and diabetes outcome.

In [ ]:
# Distribution of the target variable

plt.figure(figsize=(6, 4))
sns.countplot(x="Outcome", data=df)
plt.title("Distribution of Diabetes Outcome")
plt.xlabel("Outcome (0 = No Diabetes, 1 = Diabetes)")
plt.ylabel("Number of Patients")
plt.show()

In [ ]:
# Distribution of important numerical features

features = ["Glucose", "BloodPressure", "BMI", "Age"]

df[features].hist(figsize=(12, 8), bins=20)
plt.suptitle("Distribution of Important Features")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Analysis

plt.figure(figsize=(10, 7))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Separate Features and Target

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# Define Models

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric="logloss"
    )
}

print("Models created successfully!")

In [ ]:
# Train and Evaluate Models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():

    # Scale data for models that benefit from scaling
    if name in ["Logistic Regression", "K-Nearest Neighbors"]:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

display(results_df)

In [ ]:
# Model Comparison

results_plot = results_df.set_index("Model")

results_plot[
    ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Select Best Model Based on ROC-AUC

best_model_name = results_df.loc[
    results_df["ROC-AUC"].idxmax(),
    "Model"
]

print("Best Model:", best_model_name)

In [ ]:
# Feature Importance - Random Forest

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(importance)

## Correlation Analysis

A correlation heatmap is used to understand the relationships between different numerical features in the dataset.

It helps identify which features are strongly or weakly related to each other and can also show which features have a stronger relationship with the diabetes outcome.

In [ ]:
# Correlation heatmap

plt.figure(figsize=(10, 7))

correlation = df.corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap of Diabetes Dataset")
plt.tight_layout()
plt.show()

### Key Observations

- Glucose generally shows one of the strongest relationships with the Outcome.
- BMI and Age may also show meaningful relationships with diabetes risk.
- Some features have weak correlations with the Outcome.
- Correlation helps us understand relationships between variables before building the machine learning models.

## Data Preprocessing

Before training the machine learning models, the dataset is divided into input features (X) and the target variable (y).

The data is then split into training and testing sets so that the models can be trained on one portion of the data and evaluated on unseen data.

In [ ]:
# Separate features and target

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)

## Feature Scaling

The features in the dataset have different numerical ranges. Feature scaling is therefore applied to ensure that each feature contributes appropriately to the machine learning models.

StandardScaler is used to transform the features so that they have a mean close to 0 and a standard deviation close to 1.

In [ ]:
# Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully!")
print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)

## Machine Learning Model Training

Multiple supervised machine learning algorithms are trained on the preprocessed dataset.

The models are compared to determine which algorithm provides the best performance for early diabetes risk screening.

In [ ]:
# Import machine learning models

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Create models

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    )
}

print("Models created successfully!")
print("\nModels to be evaluated:")

for name in models:
    print("-", name)

## Model Training and Evaluation

The selected machine learning models are trained using the training dataset.

Each model is evaluated on the unseen testing dataset using accuracy, precision, recall, F1-score, and ROC-AUC score.

In [ ]:
# Train and evaluate all models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():

    # Train the model
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = model.predict(X_test_scaled)

    # Probability predictions for ROC-AUC
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Display results
results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

## Model Performance Comparison

The performance of the machine learning models is compared using accuracy, precision, recall, F1-score, and ROC-AUC.

This comparison helps identify the strengths and weaknesses of each model and determine the most suitable model for diabetes risk screening.

In [ ]:
# Visualize model performance

metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

results_plot = results_df.set_index("Model")[metrics]

results_plot.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Machine Learning Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(title="Metrics")
plt.tight_layout()
plt.show()

## Model Performance Comparison

The performance of the trained machine learning models is compared using accuracy, precision, recall, F1-score, and ROC-AUC.

This visualization helps identify the strengths and weaknesses of each model and supports the selection of the most suitable model for diabetes risk screening.

In [ ]:
# Model Performance Comparison

metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

results_plot = results_df.set_index("Model")[metrics]

results_plot.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Machine Learning Model Performance Comparison")
plt.xlabel("Models")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.legend(title="Metrics")
plt.tight_layout()
plt.show()

## Model Comparison

The performance of the trained machine learning models is compared using accuracy, precision, recall, F1-score, and ROC-AUC.

Visualizing these metrics makes it easier to identify the best-performing model for diabetes risk screening.

In [ ]:
# Model Comparison Visualization

metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

results_melted = results_df.melt(
    id_vars="Model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_melted,
    x="Model",
    y="Score",
    hue="Metric"
)

plt.title("Machine Learning Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

## Confusion Matrix Analysis

A confusion matrix is used to evaluate the classification performance of the machine learning models.

It shows the number of correct and incorrect predictions for both diabetes and non-diabetes cases.

In [ ]:
# Confusion Matrix for all models

from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        ax=axes[i],
        cmap="Blues"
    )
    
    axes[i].set_title(name)

# Hide unused subplot
axes[-1].axis("off")

plt.suptitle("Confusion Matrices of Machine Learning Models", fontsize=16)
plt.tight_layout()
plt.show()

## ROC Curve Analysis

The Receiver Operating Characteristic (ROC) curve is used to evaluate the classification performance of the machine learning models at different probability thresholds.

The Area Under the Curve (ROC-AUC) provides a single measure of how well each model distinguishes between diabetic and non-diabetic cases.

In [ ]:
# ROC Curve Comparison

plt.figure(figsize=(10, 7))

for name, model in models.items():
    
    model.fit(X_train_scaled, y_train)
    
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    
    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

# Random classifier reference line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Final Model Selection

Based on the evaluation metrics and ROC-AUC scores, the models are compared to determine the most suitable model for early diabetes risk screening.

Logistic Regression achieved the highest ROC-AUC score of 0.8230, indicating strong ability to distinguish between diabetic and non-diabetic cases.

Although Random Forest achieved slightly higher accuracy, Logistic Regression provides the best ROC-AUC performance among the evaluated models.

In [ ]:
# Final Model Selection

best_model_name = results_df.loc[
    results_df["ROC-AUC"].idxmax(), "Model"
]

best_roc_auc = results_df["ROC-AUC"].max()

print("Best Model:", best_model_name)
print("Best ROC-AUC:", round(best_roc_auc, 4))

## Feature Importance Analysis

Feature importance analysis is performed to understand which health-related variables contribute most to the diabetes risk prediction.

For the Logistic Regression model, the magnitude of the model coefficients is used to estimate the relative importance of each feature.

In [ ]:
# Feature Importance using Logistic Regression

# Train the final Logistic Regression model
final_model = LogisticRegression(max_iter=1000)

final_model.fit(X_train_scaled, y_train)

# Get feature coefficients
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": final_model.coef_[0]
})

# Calculate absolute importance
feature_importance["Absolute Importance"] = (
    feature_importance["Importance"].abs()
)

# Sort by importance
feature_importance = feature_importance.sort_values(
    by="Absolute Importance",
    ascending=False
)

display(feature_importance)

In [ ]:
# Visualize Feature Importance

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance,
    x="Absolute Importance",
    y="Feature"
)

plt.title("Feature Importance - Logistic Regression")
plt.xlabel("Absolute Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Final Diabetes Risk Prediction

The selected Logistic Regression model is used to predict the diabetes risk for a new patient based on clinical measurements.

The input values are scaled using the same StandardScaler fitted during model training.

In [ ]:
# Final Diabetes Risk Prediction

# Example patient
patient = np.array([[
    2,      # Pregnancies
    120,    # Glucose
    70,     # BloodPressure
    25,     # SkinThickness
    80,     # Insulin
    30.5,   # BMI
    0.45,   # DiabetesPedigreeFunction
    35      # Age
]])

# Scale the patient data
patient_scaled = scaler.transform(patient)

# Make prediction
prediction = final_model.predict(patient_scaled)[0]

# Get diabetes probability
probability = final_model.predict_proba(patient_scaled)[0][1]

print("SugarSense Prediction")
print("---------------------")

if prediction == 1:
    print("Prediction: Diabetes Risk Detected")
else:
    print("Prediction: Low Diabetes Risk")

print("Risk Probability:", round(probability * 100, 2), "%")

# SugarSense: Early Diabetes Risk Screening with Supervised Machine Learning

## Project Summary

SugarSense is a supervised machine learning project designed to analyze clinical measurements and perform early diabetes risk screening.

The dataset contains 768 patient records with 8 health-related features and one target variable, `Outcome`.

The project followed the following workflow:

1. Data Loading and Understanding
2. Exploratory Data Analysis
3. Correlation Analysis
4. Data Preprocessing
5. Train-Test Split
6. Feature Scaling
7. Machine Learning Model Training
8. Model Evaluation
9. Model Comparison
10. Confusion Matrix Analysis
11. ROC Curve Analysis
12. Final Model Selection
13. Feature Importance Analysis
14. Diabetes Risk Prediction

## Models Evaluated

The following supervised machine learning algorithms were evaluated:

- Logistic Regression
- Random Forest
- XGBoost
- K-Nearest Neighbors
- Decision Tree

## Final Results

Logistic Regression achieved the highest ROC-AUC score of approximately 0.823 and was selected as the final model for the SugarSense screening system.

Random Forest achieved the highest accuracy among the evaluated models.

## Conclusion

The results demonstrate that supervised machine learning can be used to analyze clinical measurements and assist in early diabetes risk screening.

Glucose, BMI, Age, and other patient-related features provide useful information for predicting diabetes risk.

SugarSense is intended as a screening and educational machine learning project and should not be considered a substitute for professional medical diagnosis.

In [ ]:
# SugarSense - Diabetes Risk Prediction Demo

# Train the final Logistic Regression model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_scaled, y_train)

print("SugarSense prediction model trained successfully!")

In [ ]:
# Patient information for prediction

patient_data = [[
    1,      # Pregnancies
    120,    # Glucose
    70,     # BloodPressure
    25,     # SkinThickness
    80,     # Insulin
    30.0,   # BMI
    0.5,    # DiabetesPedigreeFunction
    30      # Age
]]

# Convert patient data using the same scaler used during training
patient_scaled = scaler.transform(patient_data)

# Make prediction
prediction = final_model.predict(patient_scaled)[0]
probability = final_model.predict_proba(patient_scaled)[0][1]

print("SugarSense Prediction")
print("---------------------")

if prediction == 1:
    print("Prediction: Higher Diabetes Risk")
else:
    print("Prediction: Lower Diabetes Risk")

print(f"Estimated Risk Probability: {probability:.2%}")

In [ ]:
# SugarSense - Interactive Patient Prediction

print("Enter Patient Information")
print("--------------------------")

pregnancies = float(input("Pregnancies: "))
glucose = float(input("Glucose: "))
blood_pressure = float(input("Blood Pressure: "))
skin_thickness = float(input("Skin Thickness: "))
insulin = float(input("Insulin: "))
bmi = float(input("BMI: "))
diabetes_pedigree = float(input("Diabetes Pedigree Function: "))
age = float(input("Age: "))

# Create patient data
patient_data = [[
    pregnancies,
    glucose,
    blood_pressure,
    skin_thickness,
    insulin,
    bmi,
    diabetes_pedigree,
    age
]]

# Scale using the trained scaler
patient_scaled = scaler.transform(patient_data)

# Predict
prediction = final_model.predict(patient_scaled)[0]
probability = final_model.predict_proba(patient_scaled)[0][1]

print("\n===== SugarSense Result =====")

if prediction == 1:
    print("Prediction: Higher Diabetes Risk")
else:
    print("Prediction: Lower Diabetes Risk")

print(f"Estimated Risk Probability: {probability:.2%}")
print("=============================")

In [ ]:
# Save the trained model and scaler

import joblib

joblib.dump(final_model, "sugarsense_model.pkl")
joblib.dump(scaler, "sugarsense_scaler.pkl")

print("SugarSense model saved successfully!")
print("SugarSense scaler saved successfully!")

# Rubric-Compliant Final Modeling Pipeline

This section rebuilds the modeling workflow using leak-free preprocessing,
feature engineering, stratified validation, and all six required models.

In [ ]:
# ==========================================
# STEP 1: DATA AUDIT + FEATURE ENGINEERING
# ==========================================

import numpy as np
import pandas as pd

# Work from the original dataset
data = df.copy()

print("Dataset shape:", data.shape)
print("\nData types:")
print(data.dtypes)

print("\nDuplicate rows:", data.duplicated().sum())

print("\nClass balance:")
print(data["Outcome"].value_counts())
print("\nClass percentages:")
print(data["Outcome"].value_counts(normalize=True) * 100)

print("\nDescriptive statistics:")
display(data.describe())

# ------------------------------------------
# Detect impossible zero values
# ------------------------------------------

zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_summary = pd.DataFrame({
    "Column": zero_columns,
    "Zero Count": [data[col].eq(0).sum() for col in zero_columns],
    "Zero Percentage": [
        data[col].eq(0).mean() * 100 for col in zero_columns
    ]
})

display(zero_summary.round(2))

# Replace medically impossible zeros with NaN
data[zero_columns] = data[zero_columns].replace(0, np.nan)

print("\nMissing values after converting impossible zeros:")
display(data.isna().sum())

# ------------------------------------------
# Feature Engineering
# ------------------------------------------

# 1. Missing glucose measurement indicator
data["Glucose_Missing"] = data["Glucose"].isna().astype(int)

# 2. Obesity indicator based on BMI
data["Obesity_Flag"] = (data["BMI"] >= 30).astype(int)

# 3. Glucose-BMI interaction
data["Glucose_BMI_Interaction"] = data["Glucose"] * data["BMI"]

print("\nNew engineered features:")
print("1. Glucose_Missing")
print("2. Obesity_Flag")
print("3. Glucose_BMI_Interaction")

print("\nFinal dataset shape:", data.shape)

display(data.head())

## Train-Test Split and Model Pipelines

The dataset is divided into stratified training and testing sets using an 80/20 split.
All preprocessing steps are placed inside model-specific pipelines to prevent data leakage.

Scaling is used for Logistic Regression, KNN, and SVM because these algorithms are sensitive
to feature magnitude. Tree-based models do not require feature scaling.

Five-fold stratified cross-validation is used to fairly compare all six supervised learning models.

In [ ]:
# ==========================================
# STEP 2: TRAIN-TEST SPLIT + MODEL PIPELINES
# ==========================================

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# ------------------------------------------
# Separate features and target
# ------------------------------------------

X = data.drop("Outcome", axis=1)
y = data["Outcome"]

# ------------------------------------------
# Stratified 80/20 train-test split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).round(3))

# ------------------------------------------
# Calculate class imbalance ratio
# ------------------------------------------

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("\nXGBoost scale_pos_weight:", round(scale_pos_weight, 3))

# ------------------------------------------
# Common preprocessing
# ------------------------------------------

# Median imputation is performed INSIDE each pipeline.
# This prevents information from the validation/test set
# leaking into the training process.

scaled_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# ------------------------------------------
# Define six required models
# ------------------------------------------

models = {

    "Logistic Regression": Pipeline([
        ("preprocessing", scaled_preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("preprocessing", scaled_preprocessor),
        ("model", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "Decision Tree": Pipeline([
        ("preprocessing", tree_preprocessor),
        ("model", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessing", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "XGBoost": Pipeline([
        ("preprocessing", tree_preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "SVM": Pipeline([
        ("preprocessing", scaled_preprocessor),
        ("model", SVC(
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

print("\nModels prepared:")
for name in models:
    print("-", name)

# ------------------------------------------
# 5-Fold Stratified Cross-Validation
# ------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# ------------------------------------------
# Evaluate every model
# ------------------------------------------

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    train_score = scores["train_roc_auc"].mean()
    validation_score = scores["test_roc_auc"].mean()

    overfit_gap = train_score - validation_score

    cv_results.append({
        "Model": name,

        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),

        "Precision Mean": scores["test_precision"].mean(),
        "Precision Std": scores["test_precision"].std(),

        "Recall Mean": scores["test_recall"].mean(),
        "Recall Std": scores["test_recall"].std(),

        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),

        "ROC-AUC Mean": validation_score,
        "ROC-AUC Std": scores["test_roc_auc"].std(),

        "Train ROC-AUC": train_score,
        "Overfit Gap": overfit_gap
    })

# ------------------------------------------
# Create comparison table
# ------------------------------------------

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    "ROC-AUC Mean",
    ascending=False
).reset_index(drop=True)

display(cv_results_df.round(4))
from sklearn.svm import SVC

models["SVM"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVC(
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

## Final Model Evaluation

The selected Logistic Regression model was evaluated on the unseen test set using a confusion matrix and classification report. These metrics provide a detailed assessment of the model's predictive performance.s

In [ ]:
# ==========================================
# Final Model Evaluation
# ==========================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Predictions on unseen test data
y_test_pred = final_model.predict(X_test_scaled)

# Classification Report
print("===== Classification Report =====")
print(classification_report(y_test, y_test_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

print("\n===== Confusion Matrix =====")
print(cm)

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Diabetes", "Diabetes"]
).plot(ax=ax)

plt.title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()

# Conclusion

The SugarSense project developed a supervised machine learning approach for early diabetes risk screening using patient health and demographic features.

After data preprocessing, exploratory analysis, feature engineering, stratified train-test splitting, and cross-validation, six classification models were evaluated. Logistic Regression achieved the best overall cross-validation performance, with an average accuracy of approximately 77.20% and ROC-AUC of approximately 0.8436.

Glucose was identified as an important predictor of diabetes risk, while BMI and Age also showed meaningful relationships with the target outcome.

The model can be used as a screening-support tool to estimate diabetes risk from patient measurements. However, the predictions should not be considered a medical diagnosis. The dataset has limitations, including its relatively small size and the presence of missing or clinically unrealistic values represented as zeros. Further validation on larger and more diverse datasets would be required before any real-world clinical use.

Overall, SugarSense demonstrates an end-to-end supervised machine learning workflow for early diabetes risk screening, from data preparation and model development to evaluation and an interactive prediction application.

# Limitations and Ethical Considerations

- The dataset contains only 768 patient records, which limits generalizability.
- Some clinical measurements contain zero values that may represent missing or unreliable measurements.
- The model may not perform equally well across different populations.
- A false negative may cause a person with diabetes risk to be overlooked, while a false positive may cause unnecessary concern or follow-up testing.
- The system is intended for educational and screening-support purposes and must not replace professional medical diagnosis.
- Larger, more diverse, and clinically validated datasets would be required before deployment in real healthcare settings.

## Threshold Tuning for High Recall

The default classification threshold of 0.50 is not necessarily optimal for diabetes screening.

Because missing a patient who may be at risk can be more concerning than generating an additional false positive, the decision threshold is adjusted to achieve at least 85% recall.

The threshold is selected using out-of-fold predictions from the training data so that the test set remains untouched until final evaluation.

In [ ]:
# ==========================================
# Threshold Tuning for Recall >= 85%
# ==========================================

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Create a clean pipeline for threshold tuning
# Missing values are replaced with the median
# Scaling is applied before Logistic Regression

threshold_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", clone(final_model))
])

# Generate out-of-fold probability predictions
oof_probabilities = cross_val_predict(
    threshold_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Test thresholds from 0.10 to 0.90
threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):

    predictions = (oof_probabilities >= threshold).astype(int)

    recall = recall_score(
        y_train,
        predictions,
        zero_division=0
    )

    precision = precision_score(
        y_train,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        predictions,
        zero_division=0
    )

    if recall >= 0.85:
        threshold_results.append({
            "Threshold": threshold,
            "Recall": recall,
            "Precision": precision,
            "F1": f1
        })

threshold_results_df = pd.DataFrame(threshold_results)

# Select threshold with highest precision
# while maintaining Recall >= 85%

if not threshold_results_df.empty:

    best_threshold_row = threshold_results_df.loc[
        threshold_results_df["Precision"].idxmax()
    ]

    best_threshold = best_threshold_row["Threshold"]

    print("==========================================")
    print("Threshold Tuning Results")
    print("==========================================")
    print("Selected threshold:", round(best_threshold, 2))
    print("Recall:", f"{best_threshold_row['Recall']:.2%}")
    print("Precision:", f"{best_threshold_row['Precision']:.2%}")
    print("F1 Score:", f"{best_threshold_row['F1']:.2%}")

    display(threshold_results_df.round(4))

else:
    print("No threshold achieved Recall >= 85%.")

In [ ]:
# ==========================================
# Compare Default vs Tuned Threshold
# ==========================================

# Fit the threshold-tuning pipeline on training data
threshold_model.fit(X_train, y_train)

# Test-set probabilities
test_probabilities = threshold_model.predict_proba(X_test)[:, 1]

# Default threshold
default_predictions = (test_probabilities >= 0.50).astype(int)

# Tuned threshold
tuned_predictions = (test_probabilities >= best_threshold).astype(int)

# Compare performance
comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 Score"],
    "Default Threshold (0.50)": [
        precision_score(y_test, default_predictions, zero_division=0),
        recall_score(y_test, default_predictions, zero_division=0),
        f1_score(y_test, default_predictions, zero_division=0)
    ],
    "Tuned Threshold": [
        precision_score(y_test, tuned_predictions, zero_division=0),
        recall_score(y_test, tuned_predictions, zero_division=0),
        f1_score(y_test, tuned_predictions, zero_division=0)
    ]
})

display(comparison.round(4))

## Feature Importance and Model Interpretation

Feature importance is examined to understand which patient characteristics contribute most to the model's diabetes risk predictions. This helps connect the machine learning results with the patterns identified during exploratory data analysis.

## Feature Importance and Model Interpretation

The Logistic Regression coefficients are examined to identify the features that have the strongest influence on the model's predictions.

Positive coefficients indicate a higher contribution toward diabetes risk, while negative coefficients indicate a lower contribution. The absolute coefficient magnitude is used to compare the relative influence of the features.

In [ ]:
# ==========================================
# Feature Importance - Logistic Regression
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Get Logistic Regression coefficients
coefficients = np.ravel(final_model.coef_)

# Diabetes dataset feature names
feature_names = [
    "Pregnancies",
    "Glucose",
    "Blood Pressure",
    "Skin Thickness",
    "Insulin",
    "BMI",
    "Diabetes Pedigree Function",
    "Age"
]

# Safety check
print("Number of features:", len(feature_names))
print("Number of coefficients:", len(coefficients))

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Absolute coefficient = importance
feature_importance["Absolute Importance"] = (
    feature_importance["Coefficient"].abs()
)

# Sort by importance
feature_importance = feature_importance.sort_values(
    "Absolute Importance",
    ascending=False
).reset_index(drop=True)

# Display
print("\nFeature Importance - Logistic Regression")
display(feature_importance.round(4))

# ==========================================
# Feature Importance Plot
# ==========================================

plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Coefficient"]
)

plt.axvline(0, linewidth=1)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Feature Importance - Logistic Regression")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Final Model Evaluation
# ==========================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Use tuned threshold
final_predictions = (
    test_probabilities >= best_threshold
).astype(int)

# Classification report
print("==========================================")
print("Final Model Evaluation")
print("==========================================")

print(
    classification_report(
        y_test,
        final_predictions,
        target_names=["Lower Risk", "Higher Risk"],
        zero_division=0
    )
)

# Confusion Matrix
cm = confusion_matrix(y_test, final_predictions)

print("Confusion Matrix:")
print(cm)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Lower Risk", "Higher Risk"]
)

disp.plot()
plt.title("SugarSense - Confusion Matrix")
plt.tight_layout()
plt.show()

# Final Conclusions

## Project Summary

SugarSense is an early diabetes risk screening system developed using supervised machine learning. The project uses clinical and health-related measurements to estimate the probability of diabetes risk.

## Key Findings

- The dataset contains 768 patient records and 8 predictive features.
- Multiple supervised learning algorithms were evaluated, including:
  - Logistic Regression
  - K-Nearest Neighbors
  - Decision Tree
  - Random Forest
  - XGBoost
- Logistic Regression achieved the best cross-validation ROC-AUC performance of approximately 0.823.
- Threshold tuning was performed to prioritize recall, which is important for early-risk screening.
- A tuned probability threshold of 0.21 achieved approximately:
  - Recall: 85.98%
  - Precision: 52.27%
  - F1 Score: 65.02%
- Feature importance analysis was performed using Logistic Regression coefficients to improve model interpretability.

## Practical Application

The trained model was integrated into a Streamlit-based web application where users can enter patient measurements and receive an estimated diabetes risk probability.

## Limitations

- The dataset is relatively small and may not represent the entire population.
- The model is intended for educational and screening purposes only.
- The prediction should not be considered a medical diagnosis.
- Further validation using larger and more diverse clinical datasets would be required before real-world medical deployment.

## Conclusion

SugarSense demonstrates how supervised machine learning can be used to develop an interpretable early diabetes risk screening system. The combination of model comparison, cross-validation, threshold tuning, and feature interpretation provides a structured machine learning workflow from data preparation to deployment.